# Hyperparameter Tuning

Based on Phase 9, XGBoost with Class Weights is our strongest and fastest performer. We will use Optuna to fine-tune its hyperparameters on the validation set to maximize PR-AUC.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.metrics import average_precision_score
import sys
sys.path.append('..')

from src.preprocessing import load_and_clean_data, preprocess_and_split


In [2]:
data_path = '../data/raw/PS_20174392719_1491204439457_log.csv'
print("Loading Data...")
df = load_and_clean_data(data_path)
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = preprocess_and_split(df, val_size=0.15, test_size=0.15, random_state=42)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("Data ready!")

Loading Data...


Data ready!


In [3]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    preds_proba = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, preds_proba)
    
    return pr_auc

# We'll do 10 trials for time efficiency.
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print("Best Trial:")
print(study.best_trial.value)
print("Best Params:")
print(study.best_params)

[I 2026-08-15 21:55:31,404] A new study created in memory with name: no-name-50902c77-0847-4d70-af6c-df825ea40824


[I 2026-08-15 21:55:37,127] Trial 0 finished with value: 0.9960584114264833 and parameters: {'n_estimators': 135, 'max_depth': 3, 'learning_rate': 0.08988856784891536, 'subsample': 0.6236446410691399, 'colsample_bytree': 0.6966414714014663}. Best is trial 0 with value: 0.9960584114264833.


[I 2026-08-15 21:55:47,925] Trial 1 finished with value: 0.9953586412374291 and parameters: {'n_estimators': 136, 'max_depth': 8, 'learning_rate': 0.05649302991184524, 'subsample': 0.7761573683438283, 'colsample_bytree': 0.8076646708386909}. Best is trial 0 with value: 0.9960584114264833.


[I 2026-08-15 21:55:54,575] Trial 2 finished with value: 0.9953580925087477 and parameters: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.19137117000811482, 'subsample': 0.7122466382407019, 'colsample_bytree': 0.8701969754881878}. Best is trial 0 with value: 0.9960584114264833.


[I 2026-08-15 21:56:05,984] Trial 3 finished with value: 0.9949793880973083 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.0789877712310545, 'subsample': 0.826549400424935, 'colsample_bytree': 0.7411468230577827}. Best is trial 0 with value: 0.9960584114264833.


...

[I 2026-08-15 21:56:34,753] Trial 6 finished with value: 0.994617901858687 and parameters: {'n_estimators': 147, 'max_depth': 9, 'learning_rate': 0.03478357150917664, 'subsample': 0.6297643518070907, 'colsample_bytree': 0.8379255688109905}. Best is trial 4 with value: 0.9962516839318423.


[I 2026-08-15 21:56:45,523] Trial 7 finished with value: 0.9954321424754862 and parameters: {'n_estimators': 131, 'max_depth': 7, 'learning_rate': 0.05210407592239468, 'subsample': 0.9359473549226565, 'colsample_bytree': 0.882244398266395}. Best is trial 4 with value: 0.9962516839318423.


[I 2026-08-15 21:56:55,415] Trial 8 finished with value: 0.9952703176216016 and parameters: {'n_estimators': 141, 'max_depth': 8, 'learning_rate': 0.08798902373791674, 'subsample': 0.896901474525894, 'colsample_bytree': 0.7698202515895561}. Best is trial 4 with value: 0.9962516839318423.


[I 2026-08-15 21:57:02,978] Trial 9 finished with value: 0.995915193704189 and parameters: {'n_estimators': 147, 'max_depth': 4, 'learning_rate': 0.17272352788132733, 'subsample': 0.8215258576131643, 'colsample_bytree': 0.7630907528423603}. Best is trial 4 with value: 0.9962516839318423.


Best Trial:
0.9962516839318423
Best Params:
{'n_estimators': 126, 'max_depth': 5, 'learning_rate': 0.0677671238187077, 'subsample': 0.9036819393132569, 'colsample_bytree': 0.8691323646994898}


In [4]:
import json
results = {
    'best_pr_auc': study.best_trial.value,
    'best_params': study.best_params
}
with open('../reports/hyperparameter_tuning.json', 'w') as f:
    json.dump(results, f, indent=4)
print("Saved tuning results.")

Saved tuning results.
